In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["KERAS_BACKEND"] = "torch"
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from pathlib import Path
import pandas as pd
import tensorflow as tf

import keras
import torch
print(keras.backend.backend())
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
# Local files/code
import src.data_preprocessing.text_preprocessing as text_pre
import src.config as config
from src.data_preprocessing.USDA_processing import embed_USDA_data, make_USDA_plant_sheets_dataframe
from sklearn.model_selection import train_test_split
from src.agents.U_NET import UNET

torch
True
NVIDIA GeForce RTX 5070


In [ ]:
IMG_SIZE = (384,384)
BATCH_SIZE = 8

In [3]:
train_image_path = Path.cwd() / 'data' / 'plantseg' / 'images' / 'train'
train_mask_path = Path.cwd() / 'data' / 'plantseg' / 'annotations' / 'train_binarized'

test_image_path = Path.cwd() / 'data' / 'plantseg' / 'images' / 'test'
test_mask_path = Path.cwd() / 'data' / 'plantseg' / 'annotations' / 'test_binarized'

val_image_path = Path.cwd() / 'data' / 'plantseg' / 'images' / 'val'
val_mask_path = Path.cwd() / 'data' / 'plantseg' / 'annotations' / 'val_binarized'

In [4]:
def get_img_pairs(img_dir, mask_dir):
    img_p, mask_p = [], []
    for img in sorted(img_dir.iterdir()):
        mask = mask_dir / str(img.stem + '.png')
        if mask.exists():
            img_p.append(str(img))
            mask_p.append(str(mask))
        else:
            print("No Mask for {img.name}")
    return img_p, mask_p

In [5]:
def load_images(img_path, mask_path):
    img = tf.io.read_file(img_path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img,IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.io.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, IMG_SIZE, method='nearest')
    mask = tf.cast(mask>0, tf.float32)
    return img, mask

In [8]:
def make_dataset(image_dir, mask_dir):
    img, masks = get_img_pairs(image_dir, mask_dir)
    df = tf.data.Dataset.from_tensor_slices((img, masks))
    df = df.shuffle(len(img))
    df = df.map(load_images, num_parallel_calls=tf.data.AUTOTUNE)
    return df.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [9]:
train_df = make_dataset(train_image_path, train_mask_path)
val_df = make_dataset(val_image_path, val_mask_path)
test_df = make_dataset(test_image_path, test_mask_path)

In [ ]:
#build the U_Net
unet = UNET((384,384))

#build the unet
unet.build_unet(input_shape=(*IMG_SIZE, 3))
unet.compile()

c:\Users\Kyler_Code\Desktop\PlantQA\venv\lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [13]:
keras.mixed_precision.set_global_policy("mixed_float16")

In [12]:
unet.fit(train_df, val_df, epochs=50)
unet.evaluate(test_df)

Epoch 1/50
  7/706 ━━━━━━━━━━━━━━━━━━━━ 102:06:18 526s/step - iou: 0.1699 - loss: 0.1279

KeyboardInterrupt: 